In [2]:
# Make sure to install SAM first if you haven't in your current session:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-0wloaxrt
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-0wloaxrt
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done


In [5]:
!pip install monai nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.5 MB/s eta 0:00:0000:01


In [8]:
import os
import glob
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from monai.transforms import Compose, ScaleIntensityRange, Resize, EnsureType


try:
    from segment_anything import sam_model_registry
except ImportError:
    print("Please install SAM using the pip command above.")

# ==========================================
# 1. DATASET & PREPROCESSING PIPELINE
# ==========================================
class Kaggle_BTCV_Liver_Dataset(Dataset):
    def __init__(self, volume_paths, label_paths):
        self.volume_paths = volume_paths
        self.label_paths = label_paths
        self.valid_slices_map = []
        
        print("Scanning BTCV volumes for Liver (Label 6) slices...")
        for vol_idx, (v_path, l_path) in enumerate(zip(volume_paths, label_paths)):
            lab_data = nib.load(l_path).get_fdata()
            depth = lab_data.shape[2]
            
            for slice_idx in range(depth):
                lbl_slice = lab_data[:, :, slice_idx]
                if np.any(lbl_slice == 6):
                    self.valid_slices_map.append({
                        'vol_idx': vol_idx,
                        'slice_idx': slice_idx
                    })
        
        print(f"Total valid liver slices found: {len(self.valid_slices_map)}")

        # Separate transforms to prevent decimal blurring in masks
        self.img_transforms = Compose([
            ScaleIntensityRange(a_min=-200, a_max=400, b_min=0.0, b_max=1.0, clip=True),
            Resize(spatial_size=(384, 384)), 
            EnsureType(dtype=torch.float32)
        ])
        
        self.lbl_transforms = Compose([
            Resize(spatial_size=(384, 384), mode="nearest"), 
            EnsureType(dtype=torch.float32)
        ])

    def __len__(self):
        return len(self.valid_slices_map)

    def __getitem__(self, idx):
        mapping = self.valid_slices_map[idx]
        v_path = self.volume_paths[mapping['vol_idx']]
        l_path = self.label_paths[mapping['vol_idx']]
        slice_idx = mapping['slice_idx']
        
        vol_data = nib.load(v_path).dataobj
        lab_data = nib.load(l_path).dataobj
        
        img_slice = np.array(vol_data[..., slice_idx])
        lbl_slice_raw = np.array(lab_data[..., slice_idx])
        
        # Isolate liver (6 -> 1.0, else 0.0)
        lbl_slice = np.where(lbl_slice_raw == 6, 1.0, 0.0)
        
        img_slice = img_slice[np.newaxis, ...]
        lbl_slice = lbl_slice[np.newaxis, ...]
        
        img_tensor = self.img_transforms(img_slice)
        lbl_tensor = self.lbl_transforms(lbl_slice)
        
        # Convert grayscale to 3-channel for SAM
        img_tensor = img_tensor.repeat(3, 1, 1) 
        
        return img_tensor, lbl_tensor

# ==========================================
# 2. MoSE MODULE (SHAPE EXPERTS & ROUTER)
# ==========================================
class MoSE_Module(nn.Module):
    def __init__(self, embed_dim, num_experts=1024, top_k=512, h=24, w=24):
        super(MoSE_Module, self).__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Shape Dictionary
        self.shape_experts = nn.Parameter(torch.randn(num_experts, h, w))
        
        # Gating Network
        self.gating_network = nn.Sequential(
            nn.Conv2d(in_channels=embed_dim, out_channels=num_experts, kernel_size=1)
        )
        
    def forward(self, e_i):
        # Generate routing weights
        G = self.gating_network(e_i) 
        
        # Sparse Activation (Top-K)
        topk_values, topk_indices = torch.topk(torch.abs(G), self.top_k, dim=1)
        sparse_G = torch.zeros_like(G)
        sparse_G.scatter_(dim=1, index=topk_indices, src=G.gather(dim=1, index=topk_indices))
        
        # Generate Shape Map
        shape_map = torch.sum(sparse_G * self.shape_experts.unsqueeze(0), dim=1)
        shape_prompt = torch.sigmoid(shape_map)
        
        return shape_prompt, sparse_G

# ==========================================
# 3. SAM + MoSE INTEGRATION WRAPPER
# ==========================================
class Medical_SAM_MoSE(nn.Module):
    def __init__(self, sam_checkpoint_path, model_type="vit_b"):
        super().__init__()
        
        print("Loading foundation SAM model...")
        self.sam = sam_model_registry[model_type](checkpoint=sam_checkpoint_path)
        
        # Freeze SAM weights
        for param in self.sam.parameters():
            param.requires_grad = False
            
        # Initialize MoSE (h=24, w=24 matches SAM's internal ViT grid for a 384x384 image)
        self.mose = MoSE_Module(embed_dim=256, num_experts=1024, top_k=512, h=24, w=24)
        
        # Adapter to map 1-channel shape prompt to SAM's 256-channel requirement
        self.prompt_adapter = nn.Conv2d(in_channels=1, out_channels=256, kernel_size=1)

    def forward(self, batched_images):
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(batched_images)
            
        shape_prompt, sparse_weights = self.mose(image_embeddings)
        
        dense_shape_prompt = self.prompt_adapter(shape_prompt.unsqueeze(1))
        
        sparse_embeddings, _ = self.sam.prompt_encoder(
            points=None, boxes=None, masks=None
        )
        
        low_res_masks, iou_predictions = self.sam.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=self.sam.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_shape_prompt,
            multimask_output=False 
        )
        
        return low_res_masks, shape_prompt

# ==========================================
# 4. INITIALIZATION & DATALOADER SETUP
# ==========================================
if __name__ == "__main__":
    # Ensure these paths match your Kaggle environment exactly
    vol_paths = sorted(glob.glob('/kaggle/input/datasets/nguynhilonguetdhcn/btcv-data-2/01_btcv/images/*.nii'))
    lab_paths = sorted(glob.glob('/kaggle/input/datasets/nguynhilonguetdhcn/btcv-data-2/01_btcv/labels/*.nii'))
    
    train_ds = Kaggle_BTCV_Liver_Dataset(vol_paths, lab_paths)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
    
    # Note: To instantiate Medical_SAM_MoSE, you will need to download a SAM checkpoint
    path_to_checkpoint = "/kaggle/input/datasets/juanestebanberon1/sam-vit-b-01ec64-pth/sam_vit_b_01ec64.pth"
    model = Medical_SAM_MoSE(sam_checkpoint_path=path_to_checkpoint)

Scanning BTCV volumes for Liver (Label 6) slices...
Total valid liver slices found: 1542
Loading foundation SAM model...
